# ResNet50 Road Damage Classification
### Project: AI-Based Automated Road-Surface Condition Monitoring Using Vehicle-Mounted Cameras

**Dataset**: RDD2020 (Road Damage Dataset 2020)  
**Model Architecture**: ResNet50 (Pretrained on ImageNet)  
**Framework**: PyTorch & torchvision  

---  
#### Overview & Scope  
This notebook implements, trains, and evaluates the **ResNet50** deep learning architecture for multi-class road surface condition monitoring. Road surface image crops extracted from vehicle-mounted camera captures are classified into four primary damage categories:

1. **D00** - Longitudinal Crack
2. **D10** - Transverse Crack
3. **D20** - Alligator Crack
4. **D40** - Pothole

> **Note**: This notebook covers **ONLY** the ResNet50 model assignment for this project. Dataset preprocessing and held-out test splits are managed independently to prevent data leakage.

## 1. Environment & Path Setup
Robust resolution of project paths and importing core PyTorch/scikit-learn libraries alongside custom project utilities.

In [ ]:
import os
import sys
import time
import json
import pathlib
from pathlib import Path
from typing import Tuple, Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

# Robust project root resolution (supports execution from root or notebooks/)
current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import modular project components
from src.models.resnet50 import get_resnet50_model, create_resnet50
from src.utils import set_seed, get_device, create_dataloaders, IMAGENET_MEAN, IMAGENET_STD
from src.evaluate import evaluate_model, generate_classification_metrics, plot_and_save_confusion_matrix
from src.evaluate_test import run_test_evaluation

print(f"Project Root Directory: {PROJECT_ROOT}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

## 2. Central Experiment Configuration
All adjustable parameters for data loading, model architecture, training, and visualization are consolidated in this single cell for clarity and reproducibility.

In [ ]:
# ==============================================================================
# CENTRAL EXPERIMENT CONFIGURATION
# ==============================================================================

# Global Random Seed for Reproducibility
SEED = 42

# Relative Dataset and Output Paths
DATA_DIR_NAME = "data/classification"
RESULTS_DIR_NAME = "results"

# Data Transformation & Loader Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_WORKERS = 0  # Set to 0 for Windows CPU compatibility

# Model Architecture Hyperparameters
NUM_CLASSES = 4
PRETRAINED = True
FREEZE_BACKBONE = False
DROPOUT_RATE = 0.2

# Optimization & Training Hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 10
PATIENCE = 5

# Execution Control Flags
RUN_TRAINING = False            # Set to True to re-run training; False loads saved checkpoint
LOAD_BEST_CHECKPOINT = True    # Load verified best model weights checkpoint
RUN_TEST_EVALUATION = True     # Evaluate model on held-out test split
SHOW_SAMPLE_IMAGES = True      # Visualize dataset samples
NUM_SAMPLE_IMAGES = 8          # Number of dataset samples to plot
SHOW_SAMPLE_PREDICTIONS = True # Visualize model test set predictions
NUM_PREDICTION_SAMPLES = 8     # Number of test predictions to plot

# Target Class Definitions
CLASS_NAMES = ["D00", "D10", "D20", "D40"]
CLASS_DESCRIPTIONS = {
    "D00": "Longitudinal Crack",
    "D10": "Transverse Crack",
    "D20": "Alligator Crack",
    "D40": "Pothole"
}

print("Central Experiment Configuration initialized successfully.")
print(f"Target Output Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## 3. Reproducibility & Computation Device
Configure random seeds across Python, NumPy, and PyTorch and detect available hardware accelerator (CUDA GPU / CPU).

In [ ]:
# Fix seeds for exact reproducibility
set_seed(SEED)
device = get_device()

# Define absolute directory paths derived from PROJECT_ROOT
DATA_DIR = (PROJECT_ROOT / DATA_DIR_NAME).resolve()
MODELS_DIR = (PROJECT_ROOT / RESULTS_DIR_NAME / "models" / "resnet50").resolve()
FIGURES_DIR = (PROJECT_ROOT / RESULTS_DIR_NAME / "figures" / "resnet50").resolve()
TABLES_DIR = (PROJECT_ROOT / RESULTS_DIR_NAME / "tables" / "resnet50").resolve()

# Ensure output directories exist
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = MODELS_DIR / "best_model.pth"

print(f"Resolved Dataset Path: {DATA_DIR}")
print(f"Resolved Checkpoint Path: {BEST_CHECKPOINT_PATH}")

## 4. Dataset Verification & Statistics
Inspect the preprocessed classification dataset split counts across `train`, `val`, and `test` splits to confirm dataset integrity.

In [ ]:
split_counts = {}
for split in ["train", "val", "test"]:
    split_path = DATA_DIR / split
    if split_path.exists():
        counts = {}
        total = 0
        for c_name in CLASS_NAMES:
            c_dir = split_path / c_name
            cnt = len(list(c_dir.glob("*.jpg"))) + len(list(c_dir.glob("*.png")))
            counts[c_name] = cnt
            total += cnt
        counts["Total"] = total
        split_counts[split.upper()] = counts
    else:
        print(f"WARNING: Directory split '{split}' not found at {split_path}")

df_dataset_stats = pd.DataFrame(split_counts).T
print("=== PREPROCESSED DATASET SAMPLE COUNTS ===")
display(df_dataset_stats)

# Verify expected test set total count (3669)
expected_test_total = 3669
actual_test_total = split_counts.get("TEST", {}).get("Total", 0)
if actual_test_total == expected_test_total:
    print(f"--> Test set sample count verified: exactly {actual_test_total} crops.")
else:
    print(f"--> WARNING: Test set sample count is {actual_test_total} (Expected: {expected_test_total}).")

## 5. DataLoaders Initialization
Instantiate PyTorch `DataLoader` instances using standard ImageNet normalization statistics (`IMAGENET_MEAN`, `IMAGENET_STD`).

In [ ]:
dataloaders = create_dataloaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMAGE_SIZE,
)

train_loader = dataloaders["train"]
val_loader = dataloaders["val"]
test_loader = dataloaders["test"]

dataset_classes = train_loader.dataset.classes
class_to_idx = train_loader.dataset.class_to_idx

print(f"DataLoaders created successfully for splits: {list(dataloaders.keys())}")
print(f"ImageFolder Dataset Classes: {dataset_classes}")
print(f"Class to Index Mapping: {class_to_idx}")

## 6. Sample Dataset Images
Dynamically visualize sample road surface condition image crops from the training DataLoader.

In [ ]:
if SHOW_SAMPLE_IMAGES:
    inputs, targets = next(iter(train_loader))
    num_display = min(NUM_SAMPLE_IMAGES, len(inputs))
    
    fig, axes = plt.subplots(2, (num_display + 1) // 2, figsize=(14, 7))
    axes = axes.flatten()
    
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    
    for i in range(num_display):
        img = inputs[i].numpy().transpose((1, 2, 0))
        img = std * img + mean  # Un-normalize
        img = np.clip(img, 0, 1)
        
        cls_idx = targets[i].item()
        cls_code = CLASS_NAMES[cls_idx]
        cls_desc = CLASS_DESCRIPTIONS.get(cls_code, "")
        
        axes[i].imshow(img)
        axes[i].set_title(f"{cls_code}: {cls_desc}", fontsize=10, fontweight="bold")
        axes[i].axis("off")
        
    for j in range(num_display, len(axes)):
        axes[j].axis("off")
        
    plt.suptitle("Sample Road Surface Damage Crops (Training Set)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 7. ResNet50 Model Initialization
Instantiate the ImageNet pretrained ResNet50 architecture and replace its linear output head for 4 target road damage classes.

In [ ]:
model = get_resnet50_model(
    num_classes=NUM_CLASSES,
    pretrained=PRETRAINED,
    freeze_backbone=FREEZE_BACKBONE,
    dropout_rate=DROPOUT_RATE,
).to(device)

# Count network parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ResNet50 Architecture Loaded Successfully.")
print(f"  - Total Parameters:     {total_params:,}")
print(f"  - Trainable Parameters: {trainable_params:,}")

# Execute dummy forward pass verification
dummy_input = torch.randn(2, 3, 224, 224).to(device)
dummy_logits = model(dummy_input)
assert dummy_logits.shape == (2, NUM_CLASSES), f"Logits shape mismatch! Got {dummy_logits.shape}"
print(f"Forward Pass Verified: Input (2, 3, 224, 224) -> Logits Shape {dummy_logits.shape}")

## 8. Training Optimization Setup
Configure `CrossEntropyLoss`, `AdamW` optimizer, and `ReduceLROnPlateau` learning rate scheduler.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

print("Training Optimization Configured:")
print(f"  - Criterion:  CrossEntropyLoss")
print(f"  - Optimizer:  AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"  - Scheduler:  ReduceLROnPlateau (factor=0.5, patience=2)")

## 9. Model Training Execution
Conditional training section. If `RUN_TRAINING == True`, executes full model training. If `RUN_TRAINING == False`, skips training to preserve computationally expensive runs.

In [ ]:
if RUN_TRAINING:
    print("RUN_TRAINING is set to True. Starting full model training...")
    from src.train import train_one_epoch
    
    best_val_loss = float("inf")
    patience_counter = 0
    history_records = []
    
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion, device)
        duration = time.time() - t0
        
        curr_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        
        history_records.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": curr_lr,
            "epoch_time": duration,
        })
        
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | Time: {duration:.1f}s")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": val_loss,
                "best_val_acc": val_acc,
                "class_names": CLASS_NAMES,
                "seed": SEED,
            }, BEST_CHECKPOINT_PATH)
            print(f"  --> Saved best model checkpoint to {BEST_CHECKPOINT_PATH}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered at epoch {epoch}.")
                break
    
    history_df = pd.DataFrame(history_records)
    history_df.to_csv(TABLES_DIR / "training_history.csv", index=False)
else:
    print("RUN_TRAINING is set to False.")
    print("Skipping training loop. Existing verified best checkpoint will be loaded in subsequent cells.")

## 10. Training History Summary
Load and display the logged training history table (`training_history.csv`).

In [ ]:
history_csv_path = TABLES_DIR / "training_history.csv"
if history_csv_path.exists():
    history_df = pd.read_csv(history_csv_path)
    print("=== TRAINING HISTORY SUMMARY ===")
    display(history_df)
else:
    print(f"Training history file not found at: {history_csv_path}")
    history_df = None

## 11. Learning Curves Visualization
Plot per-epoch Training vs Validation Loss and Accuracy curves dynamically derived from the training history.

In [ ]:
if history_df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss Curve
    ax1.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker="o", linewidth=2)
    ax1.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker="s", linewidth=2)
    ax1.set_xlabel("Epoch", fontsize=11)
    ax1.set_ylabel("Loss", fontsize=11)
    ax1.set_title("ResNet50 Loss Curve", fontsize=13, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.6)
    
    # Accuracy Curve
    ax2.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy", marker="o", linewidth=2)
    ax2.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy", marker="s", linewidth=2)
    ax2.set_xlabel("Epoch", fontsize=11)
    ax2.set_ylabel("Accuracy", fontsize=11)
    ax2.set_title("ResNet50 Accuracy Curve", fontsize=13, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, linestyle="--", alpha=0.6)
    
    plt.tight_layout()
    
    # Save notebook-generated learning curves
    fig.savefig(FIGURES_DIR / "training_validation_learning_curves.png", dpi=300)
    plt.show()
else:
    print("History DataFrame unavailable. Skipping plot generation.")

## 12. Best Model Checkpoint Loading
Load the verified best model parameters (`best_model.pth`) selected via validation set performance.

In [ ]:
if LOAD_BEST_CHECKPOINT:
    if BEST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()
        
        saved_epoch = checkpoint.get("epoch", "N/A")
        saved_val_loss = checkpoint.get("best_val_loss", "N/A")
        saved_val_acc = checkpoint.get("best_val_acc", "N/A")
        
        print(f"Successfully loaded best ResNet50 checkpoint from: {BEST_CHECKPOINT_PATH}")
        print(f"  - Checkpoint Epoch:           {saved_epoch}")
        print(f"  - Best Validation Loss:       {saved_val_loss}")
        print(f"  - Best Validation Accuracy:   {saved_val_acc}")
    else:
        print(f"ERROR: Checkpoint file not found at {BEST_CHECKPOINT_PATH}")

## 13. Validation Set Evaluation
Evaluate the loaded ResNet50 checkpoint on the Validation split (`data/classification/val`).

In [ ]:
val_loss, val_acc, val_targets, val_preds = evaluate_model(
    model=model,
    dataloader=val_loader,
    criterion=criterion,
    device=device,
)

df_val_metrics = generate_classification_metrics(val_targets, val_preds, CLASS_NAMES)

print("=== VALIDATION SET EVALUATION METRICS ===")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc * 100:.2f}%")
display(df_val_metrics)

## 14. Final Held-Out Test Evaluation
Evaluate the loaded ResNet50 checkpoint on the **held-out Test set** (`data/classification/test`, 3,669 samples).

In [ ]:
if RUN_TEST_EVALUATION:
    test_loss, test_acc, y_true_test, y_pred_test = evaluate_model(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
    )
    
    # Calculate exact summary metrics
    labels = list(range(len(CLASS_NAMES)))
    prec_cls, rec_cls, f1_cls, supp_cls = precision_recall_fscore_support(
        y_true_test, y_pred_test, labels=labels, zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true_test, y_pred_test, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true_test, y_pred_test, average="weighted", zero_division=0
    )
    
    print("=== FINAL HELD-OUT TEST EVALUATION SUMMARY ===")
    print(f"Test Set Samples Evaluated: {len(y_true_test)} (Expected: 3669)")
    print(f"Test Loss:          {test_loss:.4f}")
    print(f"Test Accuracy:      {test_acc * 100:.2f}%")
    print(f"Macro Precision:    {macro_p * 100:.2f}%")
    print(f"Macro Recall:       {macro_r * 100:.2f}%")
    print(f"Macro F1-Score:     {macro_f1 * 100:.2f}%")
    print(f"Weighted Precision: {weighted_p * 100:.2f}%")
    print(f"Weighted Recall:    {weighted_r * 100:.2f}%")
    print(f"Weighted F1-Score:  {weighted_f1 * 100:.2f}%")
    
    assert len(y_true_test) == 3669, f"Expected 3669 test samples, got {len(y_true_test)}"

## 15. Test Set Confusion Matrix
Generate and plot the heatmap confusion matrix for the test set predictions.

In [ ]:
notebook_cm_path = FIGURES_DIR / "test_confusion_matrix_notebook.png"
cm_fig = plot_and_save_confusion_matrix(
    y_true=y_true_test,
    y_pred=y_pred_test,
    class_names=CLASS_NAMES,
    save_path=str(notebook_cm_path),
    title="ResNet50 Test Confusion Matrix (Notebook)",
)
plt.show()

## 16. Test Classification Report
Display per-class precision, recall, F1-score, and support as a formatted pandas DataFrame.

In [ ]:
report_dict_test = classification_report(
    y_true_test,
    y_pred_test,
    labels=labels,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
df_test_report = pd.DataFrame(report_dict_test).transpose()
print("=== TEST SET CLASSIFICATION REPORT ===")
display(df_test_report)

# Save notebook classification report CSV
df_test_report.to_csv(TABLES_DIR / "notebook_test_classification_report.csv")

## 17. Final Test Metric Summary Table
Construct a clean summary table of overall held-out test evaluation metrics.

In [ ]:
df_test_summary = pd.DataFrame([
    {"Metric": "Test Loss", "Value": f"{test_loss:.4f}"},
    {"Metric": "Test Accuracy", "Value": f"{test_acc * 100:.2f}%"},
    {"Metric": "Macro Precision", "Value": f"{macro_p * 100:.2f}%"},
    {"Metric": "Macro Recall", "Value": f"{macro_r * 100:.2f}%"},
    {"Metric": "Macro F1-Score", "Value": f"{macro_f1 * 100:.2f}%"},
    {"Metric": "Weighted Precision", "Value": f"{weighted_p * 100:.2f}%"},
    {"Metric": "Weighted Recall", "Value": f"{weighted_r * 100:.2f}%"},
    {"Metric": "Weighted F1-Score", "Value": f"{weighted_f1 * 100:.2f}%"},
])

print("=== HELD-OUT TEST METRICS SUMMARY ===")
display(df_test_summary)

# Save notebook test summary CSV
df_test_summary.to_csv(TABLES_DIR / "notebook_test_summary.csv", index=False)

## 18. Sample Test Predictions Visualization
Dynamically visualize sample test set predictions with ground truth vs predicted class labels.

In [ ]:
if SHOW_SAMPLE_PREDICTIONS:
    test_inputs, test_targets = next(iter(test_loader))
    num_pred_display = min(NUM_PREDICTION_SAMPLES, len(test_inputs))
    
    model.eval()
    with torch.no_grad():
        batch_logits = model(test_inputs.to(device))
        _, batch_preds = torch.max(batch_logits, 1)
        batch_preds = batch_preds.cpu()
    
    fig, axes = plt.subplots(2, (num_pred_display + 1) // 2, figsize=(14, 7))
    axes = axes.flatten()
    
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    
    for i in range(num_pred_display):
        img = test_inputs[i].numpy().transpose((1, 2, 0))
        img = std * img + mean  # Un-normalize
        img = np.clip(img, 0, 1)
        
        actual_cls = CLASS_NAMES[test_targets[i].item()]
        pred_cls = CLASS_NAMES[batch_preds[i].item()]
        is_correct = (actual_cls == pred_cls)
        
        title_color = "green" if is_correct else "red"
        status_symbol = "✓" if is_correct else "✗"
        
        axes[i].imshow(img)
        axes[i].set_title(f"{status_symbol} Actual: {actual_cls}\nPred: {pred_cls}", color=title_color, fontsize=10, fontweight="bold")
        axes[i].axis("off")
        
    for j in range(num_pred_display, len(axes)):
        axes[j].axis("off")
        
    plt.suptitle("ResNet50 Held-Out Test Set Sample Predictions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 19. Notebook Results & Experiment Summary
Summary of notebook execution outputs and verification of model baseline performance.

In [ ]:
print("=== RESNET50 EXPERIMENT NOTEBOOK SUMMARY ===")
print(f"1. Verified Checkpoint Evaluated: {BEST_CHECKPOINT_PATH}")
print(f"2. Held-Out Test Set Size:        {len(y_true_test)} samples")
print(f"3. Final Test Accuracy:            {test_acc * 100:.2f}%")
print(f"4. Final Macro F1-Score:           {macro_f1 * 100:.2f}%")
print(f"5. Final Weighted F1-Score:        {weighted_f1 * 100:.2f}%")
print("")
print("Notebook Generated Result Artifacts:")
print(f"  - Figures: {FIGURES_DIR / 'test_confusion_matrix_notebook.png'}")
print(f"  - Tables:  {TABLES_DIR / 'notebook_test_summary.csv'}")
print(f"  - Tables:  {TABLES_DIR / 'notebook_test_classification_report.csv'}")
print("===============================================")